In [ ]:
import sys
import importlib

#remove path collision s
sys.path = [p for p in sys.path if "elevation" not in p]

# Force the exact path for tif henerator
tif_path = "/kaggle/input/datasets/yellapusomeshwar/tif-generator"
if tif_path not in sys.path:
    sys.path.insert(0, tif_path)

# Import fresh
import pipeline
importlib.reload(pipeline)
from pipeline import generate_drone_dataset

print("Successfully imported generate_drone_dataset from tif generator!")

In [ ]:
!pip install -q rasterio mercantile pillow tqdm requests numpy
import sys
sys.path.insert(0, "/kaggle/input/datasets/yellapusomeshwar/tif-generator")
from pipeline import generate_drone_dataset
output = generate_drone_dataset(bbox=[83.3130, 17.7180, 83.3245, 17.7280], zoom=19, tile_size=1024, overlap=0.2, output_dir="/kaggle/working/Siripuram_Map_Tiles")
print(output) #input1--drone imagery gen

In [ ]:
# 1. Install spatial geometry dependencies
!pip install -q shapely pyproj requests

# 2. Add parcel_generator using your exact Kaggle dataset mount path
import sys
# If your dataset slug uses a hyphen or underscore, match it here:
sys.path.insert(0, "/kaggle/input/datasets/yellapusomeshwar/parcel-generator")

# 3. Generate parcels for Siripuram
from pipeline import generate_parcel_layer

parcels_out = generate_parcel_layer(
    bbox=[83.3130, 17.7180, 83.3245, 17.7280],
    output_dir="/kaggle/working/parcels"
)

print(parcels_out)#siripuram

In [ ]:
# Test Tier 2 dynamic engine on an external location
external_bbox = [78.4300, 17.4100, 78.4450, 17.4250]

output_external = generate_parcel_layer(
    bbox=external_bbox,
    output_dir="/kaggle/working/parcels_external"
)

print(output_external)#banjara hil;s 

In [ ]:
# 1. Install spatial geometry dependencies
!pip install -q shapely pyproj requests

# 2. Add parcel_generator using your exact Kaggle dataset mount path
import sys
# If your dataset slug uses a hyphen or underscore, match it here:
sys.path.insert(0, "/kaggle/input/datasets/yellapusomeshwar/parcel-generator")

# 3. Generate parcels for Mumbai (Bandra Kurla Complex - Tier 2 Fallback)
from pipeline import generate_parcel_layer

parcels_out = generate_parcel_layer(
    bbox=[72.8550, 19.0550, 72.8700, 19.0700],
    output_dir="/kaggle/working/parcels_mumbai"
)

print(parcels_out)

In [ ]:
from glob import glob
tiles = sorted(glob("/kaggle/working/drone_tiles/tiles/*.tif"))
src = tiles[0] if tiles else None
if src is None:
    import numpy as np
    from PIL import Image
    t = np.full((400, 400, 3), 40, np.uint8)
    t[30:110, 30:120] = (210, 205, 195); t[220:340, 80:220] = (210, 205, 195)
    src = "/tmp/synth_tile.png"; Image.fromarray(t).save(src)
gdf = YOLO2DFootprintExtractor().extract_footprints_from_drone_tile(src)
print(len(gdf), "footprints | source:", gdf["source"].unique().tolist())
display(gdf[["confidence", "source", "area_m2"]].head())

In [ ]:
# 1. Install dependencies
!pip install -q duckdb rasterio geopandas shapely pyproj requests numpy

# 2. Add module path dynamically (handles varying Kaggle mount structures)
import sys
from pathlib import Path

candidates = [
    "/kaggle/input/datasets/yellapusomeshwar/elevation-eng",
    "/kaggle/input/elevation-eng",
    "/kaggle/input/datasets/yellapusomeshwar/elevation-eng/elevation_engine",
    "/kaggle/input/elevation-eng/elevation_engine",
]
base = next((p for p in candidates if (Path(p) / "pipeline.py").exists()), None)
assert base, "pipeline.py not found — check !ls -R /kaggle/input"
sys.path.insert(0, str(base))

# 3. Import and execute the Elevation Engine
from pipeline import generate_elevation_layer

siripuram_bbox = [83.3130, 17.7180, 83.3245, 17.7280]

print("Booting Elevation Engine...")
print("Streaming Overture 3D footprints & Copernicus DEM (takes a minute, let it run)...")

# 4. Generate the 3D footprint layer
gdf = generate_elevation_layer(
    bbox=siripuram_bbox,
    footprints_gdf=None,
    output_dir="/kaggle/working/elevation",
)

# 5. Display the cadastral results
print(f"\n Success! Elevated {len(gdf)} 3D buildings.")
display(gdf[["building_id", "Z_base", "H_final", "floors", "Z_sub", "Z_roof", "confidence"]].head(10))

In [ ]:
!pip install -q ultralytics opencv-python-headless geopandas shapely pyproj numpy laspy

import sys
from pathlib import Path
cands = ["/kaggle/input/datasets/yellapusomeshwar/elevation-eng",
         "/kaggle/input/elevation-eng"]
base = next((p for p in cands if (Path(p) / "ai_perception_engine.py").exists()), None)
assert base, "mount missing — check !ls -R /kaggle/input"
sys.path.insert(0, base)
from ai_perception_engine import PointNet3DSemanticClassifier, YOLO2DFootprintExtractor
print("imports OK from", base)

In [ ]:
!grep -c "_list_building_parts" /kaggle/input/datasets/yellapusomeshwar/elevation-eng/overture.py

In [ ]:
!grep -c "_empty_output_gdf" /kaggle/input/datasets/yellapusomeshwar/elevation-engine/pipeline.py
!grep -c "s3_url_style='vhost'" /kaggle/input/datasets/yellapusomeshwar/elevation-engine/overture.py

In [ ]:
# 1. Install dependencies
!pip install -q laspy geopandas shapely pyproj numpy

import sys
from pathlib import Path
import geopandas as gpd

# 2. Dynamically locate lidar_engine.py across /kaggle/input
engine_matches = list(Path("/kaggle/input").rglob("lidar_engine.py"))
assert len(engine_matches) > 0, "lidar_engine.py not found in /kaggle/input"
lidar_code_dir = engine_matches[0].parent
sys.path.insert(0, str(lidar_code_dir))
print(f"Loaded lidar_engine from: {lidar_code_dir}")

from lidar_engine import extract_lidar_heights, generate_lidar_elevation_geojson

# 3. Locate point cloud files
pcd_files = list(Path("/kaggle/input").rglob("*.pcd"))
assert len(pcd_files) > 0, "No .pcd files found in /kaggle/input"

print(f"Discovered {len(pcd_files)} point cloud file(s):")
for f in pcd_files[:5]:
    print(f" - {f.name}")

# Select canonical survey PCD (prefers merged survey)
selected_pcd = next((f for f in pcd_files if "merged" in f.name.lower()), pcd_files[0])
print(f"\nTarget PCD for processing: {selected_pcd}")

# load building footprints
geojson_candidates = [
    Path("/kaggle/working/elevation/elevated_buildings.geojson"),
    Path("/kaggle/working/elevated_buildings.geojson"),
]
existing_geojson = next((p for p in geojson_candidates if p.exists()), None)

if existing_geojson:
    footprints_gdf = gpd.read_file(existing_geojson)
    print(f"Loaded {len(footprints_gdf)} footprints from {existing_geojson}")
elif "gdf" in globals() and isinstance(gdf, gpd.GeoDataFrame) and len(gdf) > 0:
    footprints_gdf = gdf.copy()
    print(f"Loaded {len(footprints_gdf)} footprints from in-memory gdf.")
else:
    raise FileNotFoundError("Footprints not found. Please run the elevation engine cell first.")

# process point cloud and output geojson
output_geojson = "/kaggle/working/lidar_elevated_buildings.geojson"

print("\nRunning LiDAR height extraction and vertical slicing...")
lidar_gdf = generate_lidar_elevation_geojson(
    cloud_path=str(selected_pcd),
    footprints_gdf=footprints_gdf,
    output_path=output_geojson
)

print(f"\nSaved output to: {output_geojson}")
print(f"Total records in output: {len(lidar_gdf)}")

cols_to_show = [c for c in ["building_id", "Z_base", "H_final", "floors", "Z_sub", "Z_roof", "confidence"] if c in lidar_gdf.columns]
display(lidar_gdf[cols_to_show].head(10))

In [ ]:
lid = sorted(Path("/kaggle/input/datasets/yellapusomeshwar/lidar-input").glob("*epsg32644.pcd"))
res = PointNet3DSemanticClassifier().classify_point_cloud_classes(str(lid[0]))
print("source:", res["source"], "| n:", res["n"], "| counts:", res["counts"])


In [ ]:
from lidar_engine import extract_lidar_heights
import geopandas as gpd
from shapely.geometry import box
E0, N0 = 745766.086, 1960654.779
fp = gpd.GeoDataFrame({"building_id": ["tower"]},
      geometry=[box(E0+54, N0+138, E0+286, N0+315)], crs="EPSG:32644")
got = extract_lidar_heights(str(lid[0]), fp)
print(got)

In [ ]:
import os
from pathlib import Path

print("Directories inside /kaggle/input:")
for item in Path("/kaggle/input").iterdir():
    print(f" - {item}")

print("\nSearching for any .pcd files across /kaggle/input:")
all_pcds = list(Path("/kaggle/input").rglob("*.pcd")) + list(Path("/kaggle/input").rglob("*.PCD"))

if all_pcds:
    print(f"Found {len(all_pcds)} point cloud files:")
    for p in all_pcds[:5]:
        print(f" -> {p}")
else:
    print("No .pcd files found anywhere under /kaggle/input.")
    print("Listing all files in harsha_lidar (if exists):")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "harsha" in root.lower() or "lidar" in root.lower():
            print(f"Folder: {root}")
            print(f"Files: {files[:10]}")